In [1]:
import pandas as pd
import json
import re

In [ ]:
# importing dataframe 
df = pd.read_csv('PortPaint_unclean.csv')
df.head()
df


In [ ]:
df.iloc[0]
df["name"][0]
print(type(df["name"][0]))
print(df["name"][0])
df.head()

In [ ]:
#Extract from ["name"] column and add to new "Sitter" Column
df["Sitter"] = df["name"].str.extract(r'"Sitter"\s*:\s*"([^"]*)"')
#add artist column
df["Artist"] = df["name"].str.extract(
    r'"Artist"\s*:\s*"([^"]*)"|'
    r'"Attributed to"\s*:\s*"([^"]*)"|'
    r'"Attribution"\s*:\s*"([^"]*)"|'
    r'"Engraver"\s*:\s*"([^"]*)"'
).bfill(axis=1).iloc[:, 0]
#extract date from date column
df["Clean_Date"] = df["date"].str.extract(r'"Date"\s*:\s*"([^"]*)"')

import re
# extract gender from indexecd_topics
df["Gender"] = df["indexed_topics"].str.extract(r'(Men|Women)', flags=re.IGNORECASE)
print("done")
df.head()


In [ ]:
#extract occupations from topic
# if the indexed_topics column has "Occupation" in it, extract the list and put it in a new column called "Occupation"
df["Occupation"] = df["indexed_topics"].str.extract(r'(Military|President|Clergy|Medicine|Transportation|Governors|Art|Government)', flags=re.IGNORECASE)
#if Occupation is NaN, fill it with "Civilian"
df["Occupation"] = df["Occupation"].fillna("Civilian")
df.head(20)
df





In [ ]:
#Drop NAs
df = df[df["Sitter"].notna()].reset_index(drop=True)
#remove dates from end of some names
df["Sitter"] = df["Sitter"].str.split(",", n=1).str[0].str.strip()
#remove extraneous info from the end of some artists
df["Artist"] = df["Artist"].str.split(",", n = 1).str[0].str.strip()
#only keep numbers in Clean_Date
df["Clean_Date"] = df["Clean_Date"].astype(str).str.extract('(\d+)').astype(int)

df["Clean_Date"]


# df.to_csv("check_dates.csv")
# df.head()
df.head()


In [ ]:
#Export here:
df.to_csv("PortPaint_Clean.csv", index=False)


In [ ]:
#summary df
sitter_counts = df["Sitter"].value_counts().reset_index()
sitter_counts.columns = ["Sitter", "Count"]
print(sitter_counts)

In [ ]:
sitter_counts.to_csv("sitter_counts_test.csv")

Take cleaned.csv and remove sitters not used for project 2

In [ ]:
clean_before_removal = pd.read_csv("cleaned.csv")
clean_before_removal.head()

In [ ]:
sitters_to_keep = ["George Washington",
    "Benjamin Franklin",
    "Thomas Jefferson",
    "John Adams",
    "Alexander Hamilton",
    "James Madison"
]
cleaned_test = clean_before_removal[clean_before_removal["Sitter"].isin(sitters_to_keep)].reset_index(drop=True)

In [ ]:
print(cleaned_test["Sitter"].value_counts())
cleaned_test

In [ ]:
cleaned_test.to_csv("cleaned_test.csv")

In [ ]:
#testing if pandas can read the data
readable = pd.read_csv("cleaned_test.csv")
readable_from_github = pd.read_csv("https://raw.githubusercontent.com/nmolnar-parsons/major-studio-1/refs/heads/main/Project_2/Data/cleaned_test.csv")

In [ ]:
#more cleaning
portpaint_editedocc = pd.read_csv('PortPaint_editedOcc.csv')
portpaint_editedocc.head()

In [ ]:
#remove thumnails NAN
portpaint_editedocc = portpaint_editedocc[portpaint_editedocc["thumbnail"].notna()].reset_index(drop=True)
#remove sitters with "unidentified"
portpaint_editedocc = portpaint_editedocc[~portpaint_editedocc["Sitter"].str.contains("unidentified", case=False, na=False)].reset_index(drop=True)
#change Gender to title case
portpaint_editedocc["Gender"] = portpaint_editedocc["Gender"].str.title()



portpaint_editedocc

In [ ]:
#export
portpaint_editedocc.to_csv("PortPaint_edited_cleaned.csv", index=False)

Further refining of dataset - getting first and last initial

In [ ]:
df_for_initials = pd.read_csv("PortPaint_withfaces.csv")
df_for_initials.head()

In [ ]:
#remove any rows with \n in Sitter
df_for_initials = df_for_initials[~df_for_initials["Sitter"].str.contains(r"\\n", na=False)].reset_index(drop=True)
df_for_initials.head()

In [ ]:
df_for_initials["first_initial"] = df_for_initials["Sitter"].str.split().str[0].str[0]
df_for_initials["last_initial"] = df_for_initials["Sitter"].str.split().str[-1].str[0]
df_for_initials.head(20)


In [ ]:
#save output
df_for_initials.to_csv("PortPaint_Use.csv", index=False)

Let's make one with ONLY the founding father's I've pulled out from my previous project.

In [ ]:
# read CSV 
revperson_df = pd.read_csv('highcount_githubLinks.csv')
revperson_df.head()

In [ ]:
#remove rows where face_url column is empty
revperson_df = revperson_df[revperson_df["face_urls"].notna()].reset_index(drop=True)

#get occupation from
# if the indexed_topics column has "Occupation" in it, extract the list and put it in a new column called "Occupation"
revperson_df["Occupation"] = revperson_df["indexed_topics"].str.extract(r'(Military|President|Clergy|Medicine|Transportation|Governors|Art|Government)', flags=re.IGNORECASE)
#if Occupation is NaN, fill it with "Civilian"
revperson_df["Occupation"] = revperson_df["Occupation"].fillna("Civilian")
#Get initials
revperson_df["first_initial"] = revperson_df["Sitter"].str.split().str[0].str[0]
revperson_df["last_initial"] = revperson_df["Sitter"].str.split().str[-1].str[0]
revperson_df.head(20)

# portrait date
revperson_df["Clean_Date"] = revperson_df["date"].str.extract(r'"Date"\s*:\s*"([^"]*)"')
revperson_df["Clean_Date"] = revperson_df["Clean_Date"].astype(str).str.extract('(\d+)').astype(int)


revperson_df.head(20)


In [ ]:
#export

revperson_df.to_csv("highcount_Use.csv", index=False)

Use hosted files and not github urls

In [2]:
df_github = pd.read_csv("highcount_Use.csv")
df_github.head()

,Unnamed: 0,collectionsURL,unitCode,EDANurl,title,creditLine,date,identifier,name,notes,...,thumbnail,Sitter,Artist,Clean_Date,Gender,Occupation,faces,face_urls,first_initial,last_initial
0,0,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_XX108A,George Washington,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1803?""}","{""Object number"": ""XX108A""}","{""Attributed to"": ""William Winstanley, born En...",NaN,...,https://ids.si.edu/ids/iiif/SAAM-XX108A_1/full...,George Washington,William Winstanley,1803,Men,President,"[[623, 441, 998, 968]]",https://github.com/nmolnar-parsons/revperiod_p...,G,W
1,3,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_2014.29.1,Thomas Jefferson Presidential Inaugural Medal,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1801""}","{""Object number"": ""2014.29.1""}","{""Engraver"": ""John Reich, born Furth, Germany ...",NaN,...,https://ids.si.edu/ids/iiif/SAAM-2014.29.1_1/f...,Thomas Jefferson,John Reich,1801,Men,President,"[[708, 633, 1024, 1054]]",https://github.com/nmolnar-parsons/revperiod_p...,T,J
2,23,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_1999.27.58,John Quincy Adams,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""early 19th century""}","{""Object number"": ""1999.27.58""}","{""Artist"": ""Unidentified"", ""Sitter"": ""John Qui...","{""Luce Center Label"": ""Son of John and Abigail...",...,https://ids.si.edu/ids/iiif/SAAM-1999.27.58_1/...,John Quincy Adams,Unidentified,19,Men,Civilian,"[[632, 530, 1011, 1044]]",https://github.com/nmolnar-parsons/revperiod_p...,J,A
3,29,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_1999.27.41,Major-General Anthony Wayne,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1795""}","{""Object number"": ""1999.27.41""}","{""Artist"": ""James Peale, born Chestertown, MD ...","{""Luce Center Label"": ""\u201cMad Anthony\u201d...",...,https://ids.si.edu/ids/iiif/SAAM-1999.27.41_1/...,Anthony Wayne,James Peale,1795,Men,Military,"[[535, 573, 952, 1141]]",https://github.com/nmolnar-parsons/revperiod_p...,A,W
4,97,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_1971.111,Mourning Piece for George Washington,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1810""}","{""Object number"": ""1971.111""}","{""Artist"": ""Enoch G. Gridley, active 1803-1818...",NaN,...,https://ids.si.edu/ids/iiif/SAAM-1971.111_1/fu...,George Washington,Enoch G. Gridley,1810,Men,Military,"[[672, 933, 763, 1074], [336, 888, 402, 968]]",https://github.com/nmolnar-parsons/revperiod_p...,G,W


In [4]:
#cut out everything between Expanded_Faces and .jpg from each link in face_urls column
df_github["file_path"] = df_github["face_urls"].str.extract(r'(Expanded_Faces/.*?\.jpg)')
df_github.head()

,Unnamed: 0,collectionsURL,unitCode,EDANurl,title,creditLine,date,identifier,name,notes,...,Sitter,Artist,Clean_Date,Gender,Occupation,faces,face_urls,first_initial,last_initial,file_path
0,0,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_XX108A,George Washington,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1803?""}","{""Object number"": ""XX108A""}","{""Attributed to"": ""William Winstanley, born En...",NaN,...,George Washington,William Winstanley,1803,Men,President,"[[623, 441, 998, 968]]",https://github.com/nmolnar-parsons/revperiod_p...,G,W,Expanded_Faces/Image_0_face_0.jpg
1,3,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_2014.29.1,Thomas Jefferson Presidential Inaugural Medal,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1801""}","{""Object number"": ""2014.29.1""}","{""Engraver"": ""John Reich, born Furth, Germany ...",NaN,...,Thomas Jefferson,John Reich,1801,Men,President,"[[708, 633, 1024, 1054]]",https://github.com/nmolnar-parsons/revperiod_p...,T,J,Expanded_Faces/Image_1_face_0.jpg
2,23,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_1999.27.58,John Quincy Adams,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""early 19th century""}","{""Object number"": ""1999.27.58""}","{""Artist"": ""Unidentified"", ""Sitter"": ""John Qui...","{""Luce Center Label"": ""Son of John and Abigail...",...,John Quincy Adams,Unidentified,19,Men,Civilian,"[[632, 530, 1011, 1044]]",https://github.com/nmolnar-parsons/revperiod_p...,J,A,Expanded_Faces/Image_2_face_0.jpg
3,29,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_1999.27.41,Major-General Anthony Wayne,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1795""}","{""Object number"": ""1999.27.41""}","{""Artist"": ""James Peale, born Chestertown, MD ...","{""Luce Center Label"": ""\u201cMad Anthony\u201d...",...,Anthony Wayne,James Peale,1795,Men,Military,"[[535, 573, 952, 1141]]",https://github.com/nmolnar-parsons/revperiod_p...,A,W,Expanded_Faces/Image_3_face_0.jpg
4,97,https://collections.si.edu/search/detail/edanm...,SAAM,edanmdm:saam_1971.111,Mourning Piece for George Washington,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1810""}","{""Object number"": ""1971.111""}","{""Artist"": ""Enoch G. Gridley, active 1803-1818...",NaN,...,George Washington,Enoch G. Gridley,1810,Men,Military,"[[672, 933, 763, 1074], [336, 888, 402, 968]]",https://github.com/nmolnar-parsons/revperiod_p...,G,W,Expanded_Faces/Image_4_face_0.jpg


In [5]:
df_github.to_csv("highcount_Use_hosted.csv", index=False)